# 02 - Demo Color Correlogram

Notebook nay minh hoa cach Color Correlogram hoat dong:
- Luong tu hoa mau sac
- Tinh Auto-Correlogram
- Truc quan hoa vector dac trung
- So sanh voi Color Histogram

In [ ]:
import os, sys, cv2, time
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
from preprocessing import load_image, convert_to_hsv, quantize_colors_hsv
from color_correlogram import auto_correlogram, auto_correlogram_fast
from color_histogram import color_histogram

## 1. Doc va hien thi 1 anh mau

In [ ]:
# Doc 1 anh mau
DATA_DIR = os.path.join('..', 'data', 'corel-1k')
# Tim anh dau tien trong dataset
sample_img = None
for class_dir in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_dir)
    if os.path.isdir(class_path):
        for f in sorted(os.listdir(class_path)):
            if f.lower().endswith(('.jpg', '.png', '.bmp', '.jpeg')):
                sample_img = load_image(os.path.join(class_path, f))
                print(f'Da doc: {class_dir}/{f}')
                break
    if sample_img is not None:
        break

if sample_img is None:
    print('Khong tim thay anh! Tao anh test...')
    sample_img = np.random.randint(0, 256, (128, 128, 3), dtype=np.uint8)

plt.figure(figsize=(5, 5))
plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
plt.title('Anh goc', fontsize=14)
plt.axis('off')
plt.show()
print(f'Kich thuoc: {sample_img.shape}')

## 2. Luong tu hoa mau sac (Color Quantization)

In [ ]:
# Chuyen sang HSV va luong tu hoa
img_hsv = convert_to_hsv(sample_img)
quantized, n_colors = quantize_colors_hsv(img_hsv, h_bins=8, s_bins=3, v_bins=3)

print(f'So mau sau luong tu hoa: {n_colors}')
print(f'So mau thuc te xuat hien: {len(np.unique(quantized))}')

# Hien thi anh da luong tu hoa
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Anh goc (16M mau)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(quantized, cmap='nipy_spectral')
axes[1].set_title(f'Anh da luong tu hoa ({n_colors} mau)', fontsize=12)
axes[1].axis('off')

# Histogram cua anh luong tu hoa
axes[2].hist(quantized.ravel(), bins=n_colors, range=(0, n_colors),
             color='steelblue', edgecolor='black', alpha=0.7)
axes[2].set_title('Phan bo mau sau luong tu hoa', fontsize=12)
axes[2].set_xlabel('Ma mau')
axes[2].set_ylabel('So pixel')

plt.suptitle('Qua trinh luong tu hoa mau sac', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Tinh Color Correlogram

In [ ]:
distances = [1, 3, 5, 7]

# Tinh correlogram
start = time.time()
corr_feat = auto_correlogram_fast(quantized, n_colors, distances)
t_corr = time.time() - start

# Tinh histogram (de so sanh)
start = time.time()
hist_feat = color_histogram(quantized, n_colors)
t_hist = time.time() - start

print(f'Correlogram: {corr_feat.shape} chieu, {t_corr:.3f}s')
print(f'Histogram:   {hist_feat.shape} chieu, {t_hist:.3f}s')
print(f'\nCorrelogram lon hon Histogram {corr_feat.shape[0] / hist_feat.shape[0]:.0f}x')
print(f'Vi correlogram co {len(distances)} distances x {n_colors} mau = {len(distances) * n_colors} chieu')

In [ ]:
# Truc quan hoa vector dac trung Correlogram
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i, d in enumerate(distances):
    ax = axes[i // 2, i % 2]
    start_idx = i * n_colors
    end_idx = (i + 1) * n_colors
    values = corr_feat[start_idx:end_idx]
    
    ax.bar(range(n_colors), values, color='coral', edgecolor='black', alpha=0.7, width=1)
    ax.set_title(f'Auto-Correlogram tai d={d}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Ma mau (color code)')
    ax.set_ylabel('Xac suat alpha(c, d)')
    ax.set_xlim(-1, n_colors)

plt.suptitle('Vector dac trung Color Correlogram', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. So sanh Correlogram vs Histogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].bar(range(n_colors), hist_feat, color='steelblue', edgecolor='black', alpha=0.7, width=1)
axes[0].set_title('Color Histogram', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Ma mau')
axes[0].set_ylabel('Tan suat')
axes[0].set_xlim(-1, n_colors)

# Correlogram (d=1)
axes[1].bar(range(n_colors), corr_feat[:n_colors], color='coral', edgecolor='black', alpha=0.7, width=1)
axes[1].set_title('Color Correlogram (d=1)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Ma mau')
axes[1].set_ylabel('Xac suat alpha(c, 1)')
axes[1].set_xlim(-1, n_colors)

plt.suptitle('So sanh: Histogram chi co tan suat, Correlogram co them quan he khong gian',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nNhan xet:')
print('- Histogram: chi cho biet mau nao xuat hien nhieu/it')
print('- Correlogram: cho biet them cac pixel cung mau co nam gan nhau khong')
print('- Correlogram co nhieu thong tin hon -> ky vong phan loai tot hon')

## 5. So sanh Correlogram giua 2 lop khac nhau

In [ ]:
# Doc 1 anh tu 2 lop khac nhau
class_dirs = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])

if len(class_dirs) >= 2:
    imgs = []
    names = []
    for cls in [class_dirs[0], class_dirs[-1]]:  # Lop dau va lop cuoi
        cls_path = os.path.join(DATA_DIR, cls)
        for f in sorted(os.listdir(cls_path)):
            if f.lower().endswith(('.jpg', '.png', '.bmp')):
                imgs.append(load_image(os.path.join(cls_path, f)))
                names.append(cls)
                break
    
    if len(imgs) == 2:
        fig, axes = plt.subplots(2, 3, figsize=(15, 8))
        
        for i in range(2):
            # Hien thi anh
            axes[i, 0].imshow(cv2.cvtColor(imgs[i], cv2.COLOR_BGR2RGB))
            axes[i, 0].set_title(f'Lop: {names[i]}', fontsize=12, fontweight='bold')
            axes[i, 0].axis('off')
            
            # Tinh correlogram
            hsv = convert_to_hsv(imgs[i])
            q, nc = quantize_colors_hsv(hsv)
            feat = auto_correlogram_fast(q, nc, [1, 3, 5, 7])
            
            # Histogram
            h_feat = color_histogram(q, nc)
            axes[i, 1].bar(range(nc), h_feat, color='steelblue', alpha=0.7, width=1)
            axes[i, 1].set_title('Histogram', fontsize=11)
            axes[i, 1].set_xlim(-1, nc)
            
            # Correlogram d=1
            axes[i, 2].bar(range(nc), feat[:nc], color='coral', alpha=0.7, width=1)
            axes[i, 2].set_title('Correlogram (d=1)', fontsize=11)
            axes[i, 2].set_xlim(-1, nc)
        
        plt.suptitle('So sanh dac trung giua 2 lop khac nhau', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        print('Nhan xet: Vector dac trung cua 2 lop khac nhau co su khac biet ro rang,')
        print('dieu nay giup mo hinh hoc may phan biet duoc cac lop.')